# P97 — Un sistema de control por capas robusto para un robot móvil

## 1. Título y paper

**Paper:** *A Robust Layered Control System for a Mobile Robot*  
**Autoría:** Rodney A. Brooks  
**Año y venue:** 1986 · IEEE Journal of Robotics and Automation, 2(1), 14–23  
**Nivel:** L2 · **Motor:** `subsuncion`  
**Ficha completa:** [`P97_subsuncion`](../../papers/foundational/P97_subsuncion/README.md)

**Hito:** Demuestra que un robot puede comportarse de forma competente sin modelo del mundo, sin planificador y sin representación central.

- [doi:10.1109/JRA.1986.1087032](https://doi.org/10.1109/JRA.1986.1087032)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La arquitectura percibir-planificar-actuar construye un modelo del mundo, planifica sobre él y ejecuta el plan. Mantener ese modelo es caro, y cuando el mundo cambia a mitad de la ejecución, el plan se vuelve peligroso en vez de inútil.
2. Ejecutar una implementación mínima de la propuesta: Descomponer por comportamientos y no por funciones. Cada capa conecta percepción con acción de forma directa, y las capas inferiores —evitar obstáculos— pueden **subsumir** a las superiores. No hay representación compartida.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P58
- Walter (1950), las tortugas cibernéticas


## 4. Intuición

La arquitectura clásica construye un modelo del mundo, planifica sobre él y ejecuta el plan. Brooks propone lo contrario: capas de reflejos conectadas directamente de sensor a actuador, sin modelo y sin plan. Y construye robots que funcionan mejor.


## 5. Concepto mínimo

```text
Percibir → Modelar → Planificar → Ejecutar      ← clásico: una tubería

capa 2:  explorar
capa 1:  avanzar          ← cada capa conecta percepción con acción
capa 0:  evitar choques   ← y puede SUBSUMIR a las de arriba

Sin representación central. Sin plan. Sin estado compartido.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('subsuncion', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Llegan los dos al final con el mapa correcto?
2. ¿Cuánto estado interno guarda cada uno?
3. ¿Qué pasa si aparece un obstáculo que no estaba en el mapa?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('subsuncion', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('subsuncion', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Con el mapa correcto los dos llegan sin chocar. La diferencia está en el coste: la subsunción guarda **cero** elementos de estado interno y el planificador necesita un mapa de tres obstáculos. Y al aparecer un obstáculo no previsto, el planificador choca **1 vez** y el reactivo, **0**.


## 10. Comentario pedagógico

«El mundo es su propio mejor modelo». La frase suena a eslogan y es un argumento de ingeniería: mantener una representación actualizada cuesta, y equivocarse en ella cuesta más que no tenerla. Ese razonamiento reaparece cada vez que un agente actúa sobre un entorno que cambia mientras piensa.


## 11. Error o anti-patrón deliberado

Anti-patrón: leer el artículo como si dijera que planificar es un error.


In [ ]:
print('Brooks no dice que planificar sea malo: dice que la tuberia clasica es fragil')
print('cuando el mundo cambia entre el modelado y la ejecucion.')
print('La propia comunidad acabo en arquitecturas HIBRIDAS: capa reactiva + capa deliberativa.')

## 12. Corrección

La comparación honesta, con el planificador replanificando:


In [ ]:
r = run_paper_lab('subsuncion', seed=7)['result']
print('mundo conocido -> subsuncion:', r['subsuncion_mundo_conocido']['colisiones'],
      '| planificador:', r['planificador_mundo_conocido']['colisiones'])
print('mundo cambiado -> subsuncion:', r['subsuncion_mundo_cambiado']['colisiones'],
      '| planificador:', r['planificador_mundo_cambiado']['colisiones'])
print('Si el planificador REPLANIFICA al detectar la discrepancia, empata.')

## 13. Desafío guiado

Sigue la traza de la subsunción y localiza dónde la capa 0 subsume a la capa 1.


In [ ]:
r = run_paper_lab('subsuncion', seed=3)['result']
show(r)

## 14. Desafío autónomo

Diseña la descomposición por capas de un agente tuyo: qué comportamiento va en cada capa y cuál puede interrumpir a cuál. Después compáralo con la descomposición funcional que tenía.


## 15. Evidencia de aprendizaje

Guarda la comparación en los dos mundos y tu explicación de qué cuesta mantener un modelo.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P97_subsuncion/README.md) · evaluación formal: [`assessments/papers/P97_subsuncion.md`](../../assessments/papers/P97_subsuncion.md)


## 16. Cierre

El robot ya reacciona. Para ir a un sitio concreto en un espacio con muchos grados de libertad hace falta algo más que reflejos.


## 17. Conexión con el siguiente hito

- P102
- P106

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
